In [1]:
import pandas as pd
import ast
import re

companies = pd.read_csv("../../nyse_nasdaq_companies_all.csv")
article_company_index = pd.read_csv("../../article_company_index.csv")

companies["company_id"] = companies["company_id"].astype(str).str.strip()
companies["company"] = companies["company"].astype(str).str.strip()

def parse_company_list(x):
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    try:
        value = ast.literal_eval(str(x))
        if isinstance(value, list):
            return value
        return []
    except Exception:
        return []

article_company_index["raw_company_list"] = article_company_index["company"].apply(parse_company_list)

In [2]:
LEGAL_SUFFIXES = r"""
inc|inc\.|corporation|corp|corp\.|company|co|co\.|group|plc|
ltd|ltd\.|limited|holdings|holding|sa|s\.a\.|ag|nv|n\.v\.|
se|spa|s\.p\.a\.|asa|ab|gmbh|lp|llp|llc|l\.l\.c\.|bv|b\.v\.
"""

suffix_re = re.compile(rf"\b({LEGAL_SUFFIXES})\b", re.IGNORECASE | re.VERBOSE)

def normalize_name(x):
    if pd.isna(x):
        return None

    x = str(x).strip()
    if not x:
        return None

    x = x.replace("’s", "").replace("'s", "")
    x = x.replace("’", "'")
    x = x.lower()
    x = x.replace("&", " and ")
    x = suffix_re.sub(" ", x)
    x = re.sub(r"[^a-z0-9 ]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x or None

alias_rows = []

for _, row in companies.iterrows():
    alias = normalize_name(row["company"])

    if alias is None:
        continue

    if len(alias) < 3:
        continue

    alias_rows.append({
        "alias": alias,
        "company_id": row["company_id"],
        "company": row["company"]
    })

alias_df = pd.DataFrame(alias_rows).drop_duplicates()

alias_counts = alias_df.groupby("alias")["company_id"].nunique()
ambiguous_aliases = set(alias_counts[alias_counts > 1].index)

alias_df_clean = alias_df[~alias_df["alias"].isin(ambiguous_aliases)].copy()

alias_to_company_id = dict(zip(alias_df_clean["alias"], alias_df_clean["company_id"]))
company_id_to_name = dict(zip(companies["company_id"], companies["company"]))

In [3]:
def match_company_mentions(raw_company_list):
    matched = []

    for name in raw_company_list:
        alias = normalize_name(name)

        if alias in alias_to_company_id:
            matched.append(alias_to_company_id[alias])

    # deduplicate while preserving order
    seen = set()
    unique = []

    for cid in matched:
        if cid not in seen:
            seen.add(cid)
            unique.append(cid)

    return unique

article_company_index["matched_company_ids"] = article_company_index["raw_company_list"].apply(
    match_company_mentions
)

article_company_index["matched_companies"] = article_company_index["matched_company_ids"].apply(
    lambda ids: [company_id_to_name.get(cid, cid) for cid in ids]
)

print("Articles:", len(article_company_index))
print("Articles with raw ORG mentions:", (article_company_index["raw_company_list"].apply(len) > 0).sum())
print("Articles with matched companies:", (article_company_index["matched_company_ids"].apply(len) > 0).sum())
print("Articles with 2+ matched companies:", (article_company_index["matched_company_ids"].apply(len) >= 2).sum())

Articles: 48259
Articles with raw ORG mentions: 48259
Articles with matched companies: 15969
Articles with 2+ matched companies: 5616


In [4]:
article_company_index[
    ["article_id", "date", "title", "url", "matched_company_ids", "matched_companies"]
].to_csv("article_company_matches_clean.csv", index=False)

In [5]:
articles = pd.read_csv("../../dataset.csv")

# Make sure article_id exists
if "article_id" not in articles.columns:
    articles = articles.reset_index().rename(columns={"index": "article_id"})

matched_articles = article_company_index[
    article_company_index["matched_company_ids"].apply(len) > 0
].copy()

articles_to_process = articles.merge(
    matched_articles[["article_id", "matched_company_ids", "matched_companies"]],
    on="article_id",
    how="inner"
)

print("All articles:", len(articles))
print("Articles with matched companies:", len(articles_to_process))

All articles: 50000
Articles with matched companies: 15969


In [12]:
import spacy
nlp = spacy.load("en_core_web_sm")
EVENT_PATTERNS = {
    "acquisition_merger": [
        "acquired",
        "acquisition",
        "merger",
        "merged",
        "takeover",
        "take over",
        "proposed merger",
        "pending merger"
    ],

    "lawsuit_legal": [
        "lawsuit",
        "litigation",
        "sued",
        "settlement",
        "legal settlement",
        "charged",
        "accused",
        "probe",
        "investigation",
        "dispute"
    ],

    "earnings_results": [
        "financial results",
        "quarterly results",
        "first quarter results",
        "second quarter results",
        "third quarter results",
        "fourth quarter results",
        "reported financial results",
        "reported quarterly results",
        "reported earnings",
        "reports financial",
        "announced financial results",
        "net income",
        "net loss",
        "earnings per share",
        "loss per share",
        "adjusted ebitda",
        "revenue increased",
        "revenue decreased",
        "sales increased",
        "sales decreased",
        "earnings beat",
        "beats earnings",
        "missed earnings"
    ],

    "downgrade": [
        "downgraded",
        "downgrade",
        "cut rating",
        "lowered rating",
        "price target cut",
        "cut price target"
    ],

    "layoffs_restructuring": [
        "layoffs",
        "laid off",
        "job cuts",
        "cut jobs",
        "reduce workforce",
        "workforce reduction",
        "restructuring",
        "restructure",
        "right-sizing",
        "cost-cutting",
        "cost cutting",
        "management shake-up",
        "shake-up"
    ],

    "partnership_contract": [
        "announced a partnership",
        "entered into a partnership",
        "strategic partnership",
        "partnered with",
        "collaboration with",
        "joint venture",
        "agreement with",
        "contract from",
        "awarded contract",
        "signed a deal",
        "signed agreement"
    ],

    "dividend": [
        "declared a regular quarterly cash dividend",
        "declared a dividend",
        "declares dividend",
        "quarterly dividend",
        "cash dividend",
        "dividend will be paid"
    ],

    "ipo_listing": [
        "will float",
        "to float",
        "public offering",
        "initial public offering",
        "ipo",
        "list on",
        "admission expected"
    ],

    "regulatory_approval": [
        "regulatory approval",
        "fda approval",
        "ema approval",
        "marketing authorization"
    ]
}
def trigger_in_sentence(trigger, sentence):
    s = sentence.lower()
    trigger = trigger.lower()

    return re.search(rf"\b{re.escape(trigger)}\b", s) is not None


def detect_event_types(sentence):
    hits = []

    for event_type, triggers in EVENT_PATTERNS.items():
        for trigger in triggers:
            if trigger_in_sentence(trigger, sentence):
                hits.append((event_type, trigger))
                break

    return hits

In [13]:
def event_context_is_valid(event_type, trigger, sentence):
    s = sentence.lower()

    if event_type == "earnings_results":
        bad_phrases = [
            "reuters reported",
            "spiegel reported",
            "bbc reported",
            "fortune reported",
            "newspaper reported",
            "magazine reported",
            "media reported",
            "according to media reports"
        ]
        if any(x in s for x in bad_phrases):
            return False

    if event_type == "partnership_contract":
        bad_phrases = [
            "regional partners",
            "government partners",
            "partners had decided",
            "new partners toro rosso"
        ]
        if any(x in s for x in bad_phrases):
            return False

    if event_type == "acquisition_merger":
        bad_phrases = [
            "bought deal",
            "acquire streaming rights",
            "acquire rights",
            "acquire shares",
            "acquire stock",
            "acquire a stake",
            "bought a stake",
            "since the 2011 takeover",
            "since the takeover"
        ]
        if any(x in s for x in bad_phrases):
            return False

    return True

In [14]:
def company_appears_in_sentence(sentence, company_name):
    s = normalize_name(sentence)
    c = normalize_name(company_name)

    if not s or not c:
        return False

    return re.search(rf"\b{re.escape(c)}\b", s) is not None

TEXT_COL="text"
def extract_events_from_matched_article(row):
    article_id = row["article_id"]
    text = str(row[TEXT_COL])

    article_company_ids = row["matched_company_ids"]
    article_companies = row["matched_companies"]

    events = []
    doc = nlp(text)

    for sent in doc.sents:
        sentence = sent.text.strip()

        if len(sentence) < 20:
            continue

        hits = detect_event_types(sentence)

        if not hits:
            continue

        sentence_company_ids = []
        sentence_companies = []

        for cid, cname in zip(article_company_ids, article_companies):
            if company_appears_in_sentence(sentence, cname):
                sentence_company_ids.append(cid)
                sentence_companies.append(cname)

        for event_type, trigger in hits:
            if not event_context_is_valid(event_type, trigger, sentence):
                continue

            if sentence_company_ids:
                linked_company_ids = sentence_company_ids
                linked_companies = sentence_companies
                confidence = "high_sentence_level"
            else:
                linked_company_ids = article_company_ids
                linked_companies = article_companies
                confidence = "medium_article_level"

            events.append({
                "article_id": article_id,
                "date": row.get("date"),
                "title": row.get("title"),
                "url": row.get("url"),
                "event_type": event_type,
                "trigger": trigger,
                "linked_company_ids": linked_company_ids,
                "linked_companies": linked_companies,
                "article_company_ids": article_company_ids,
                "article_companies": article_companies,
                "sentence_company_ids": sentence_company_ids,
                "sentence_companies": sentence_companies,
                "confidence": confidence,
                "sentence": sentence
            })

    return events

In [ ]:
all_events = []

for i, row in articles_to_process.iterrows():
    if i % 500 == 0:
        print(f"Processing {i} / {len(articles_to_process)}")

    all_events.extend(extract_events_from_matched_article(row))

events_df = pd.DataFrame(all_events)

print("Events found:", len(events_df))

events_df.to_csv("company_event_candidates.csv", index=False)

events_df.head(20)

Processing 0 / 15969
Processing 500 / 15969
Processing 1000 / 15969
Processing 1500 / 15969
Processing 2000 / 15969
Processing 2500 / 15969
Processing 3000 / 15969
Processing 3500 / 15969
Processing 4000 / 15969
Processing 4500 / 15969
Processing 5000 / 15969
Processing 5500 / 15969
Processing 6000 / 15969
Processing 6500 / 15969
Processing 7000 / 15969
Processing 7500 / 15969
Processing 8000 / 15969
Processing 8500 / 15969
Processing 9000 / 15969
Processing 9500 / 15969
Processing 10000 / 15969
Processing 10500 / 15969
Processing 11000 / 15969
Processing 11500 / 15969
Processing 12000 / 15969
Processing 12500 / 15969
Processing 13000 / 15969
Processing 13500 / 15969
Processing 14000 / 15969
Processing 14500 / 15969
Processing 15000 / 15969
Processing 15500 / 15969
Events found: 40577


,article_id,date,title,url,event_type,trigger,linked_company_ids,linked_companies,article_company_ids,article_companies,sentence_company_ids,sentence_companies,confidence,sentence
0,0,2018-04-26,"McLaren review F1 technical operations, Goss m...",https://uk.reuters.com/article/uk-motor-f1-aze...,layoffs_restructuring,management shake-up,[Q9584],[Honda],[Q9584],[Honda],[],[],medium_article_level,McLaren announced a management shake-up earlie...
1,2,2018-04-10,UPDATE 2-Vitol's African venture Vivo to float...,https://www.reuters.com/article/vivo-energy-ip...,earnings_results,adjusted ebitda,[Q1428656],[Vivo],"[Q154950, Q1428656, Q219508, Q372657]","[Shell, Vivo, Citigroup, Credit Suisse]",[Q1428656],[Vivo],high_sentence_level,"April 10, 2018 / 10:27 AM / in 3 hours UPDATE ..."
2,2,2018-04-10,UPDATE 2-Vitol's African venture Vivo to float...,https://www.reuters.com/article/vivo-energy-ip...,ipo_listing,to float,[Q1428656],[Vivo],"[Q154950, Q1428656, Q219508, Q372657]","[Shell, Vivo, Citigroup, Credit Suisse]",[Q1428656],[Vivo],high_sentence_level,"April 10, 2018 / 10:27 AM / in 3 hours UPDATE ..."
3,2,2018-04-10,UPDATE 2-Vitol's African venture Vivo to float...,https://www.reuters.com/article/vivo-energy-ip...,ipo_listing,admission expected,"[Q154950, Q1428656, Q219508, Q372657]","[Shell, Vivo, Citigroup, Credit Suisse]","[Q154950, Q1428656, Q219508, Q372657]","[Shell, Vivo, Citigroup, Credit Suisse]",[],[],medium_article_level,Admission expected in May \n*
4,2,2018-04-10,UPDATE 2-Vitol's African venture Vivo to float...,https://www.reuters.com/article/vivo-energy-ip...,ipo_listing,will float,[Q1428656],[Vivo],"[Q154950, Q1428656, Q219508, Q372657]","[Shell, Vivo, Citigroup, Credit Suisse]",[Q1428656],[Vivo],high_sentence_level,By Dasha Afanasieva and Libby George \nApril 1...
5,2,2018-04-10,UPDATE 2-Vitol's African venture Vivo to float...,https://www.reuters.com/article/vivo-energy-ip...,ipo_listing,public offering,[Q154950],[Shell],"[Q154950, Q1428656, Q219508, Q372657]","[Shell, Vivo, Citigroup, Credit Suisse]",[Q154950],[Shell],high_sentence_level,"The public offering of the company, which sell..."
6,2,2018-04-10,UPDATE 2-Vitol's African venture Vivo to float...,https://www.reuters.com/article/vivo-energy-ip...,ipo_listing,to float,"[Q154950, Q1428656, Q219508, Q372657]","[Shell, Vivo, Citigroup, Credit Suisse]","[Q154950, Q1428656, Q219508, Q372657]","[Shell, Vivo, Citigroup, Credit Suisse]",[],[],medium_article_level,Vitol’s European refining and downstream ventu...
7,2,2018-04-10,UPDATE 2-Vitol's African venture Vivo to float...,https://www.reuters.com/article/vivo-energy-ip...,acquisition_merger,acquisition,[Q1428656],[Vivo],"[Q154950, Q1428656, Q219508, Q372657]","[Shell, Vivo, Citigroup, Credit Suisse]",[Q1428656],[Vivo],high_sentence_level,After the completion of an acquisition later t...
8,6,2018-05-04,Alaska Air Group Declares Quarterly Dividend,http://www.cnbc.com/2018/05/04/pr-newswire-ala...,dividend,declared a regular quarterly cash dividend,[Q4033665],[Alaska Air Group],"[Q4033665, Q822614, Q645084]","[Alaska Air Group, Alaska Airlines, Virgin Ame...",[Q4033665],[Alaska Air Group],high_sentence_level,The board of directors of Alaska Air Group (NY...
9,6,2018-05-04,Alaska Air Group Declares Quarterly Dividend,http://www.cnbc.com/2018/05/04/pr-newswire-ala...,dividend,dividend will be paid,"[Q4033665, Q822614, Q645084]","[Alaska Air Group, Alaska Airlines, Virgin Ame...","[Q4033665, Q822614, Q645084]","[Alaska Air Group, Alaska Airlines, Virgin Ame...",[],[],medium_article_level,The dividend will be paid on June 7 to all sha...


In [17]:
import ast
import pandas as pd

def parse_list_column(x):
    if isinstance(x, list):
        return x

    if pd.isna(x):
        return []

    try:
        value = ast.literal_eval(str(x))
        if isinstance(value, list):
            return value
        return []
    except Exception:
        return []


# Make sure list-like columns are actual Python lists
list_cols = [
    "linked_company_ids",
    "linked_companies",
    "article_company_ids",
    "article_companies",
    "sentence_company_ids",
    "sentence_companies"
]

for col in list_cols:
    if col in events_df.columns:
        events_df[col] = events_df[col].apply(parse_list_column)


# Summary of raw candidate events
print("Raw candidate events:", len(events_df))

print("\nEvent type counts:")
print(events_df["event_type"].value_counts())

print("\nConfidence counts:")
print(events_df["confidence"].value_counts())

print("\nEvent type by confidence:")
print(
    events_df.groupby(["event_type", "confidence"])
    .size()
    .reset_index(name="count")
    .sort_values(["event_type", "confidence"])
)

Raw candidate events: 40577

Event type counts:
event_type
earnings_results         15634
acquisition_merger       10004
lawsuit_legal             7508
layoffs_restructuring     2396
partnership_contract      1837
ipo_listing               1748
dividend                   882
regulatory_approval        358
downgrade                  210
Name: count, dtype: int64

Confidence counts:
confidence
medium_article_level    29405
high_sentence_level     11172
Name: count, dtype: int64

Event type by confidence:
               event_type            confidence  count
0      acquisition_merger   high_sentence_level   3256
1      acquisition_merger  medium_article_level   6748
2                dividend   high_sentence_level    414
3                dividend  medium_article_level    468
4               downgrade   high_sentence_level     85
5               downgrade  medium_article_level    125
6        earnings_results   high_sentence_level   3726
7        earnings_results  medium_article_level  119

In [ ]:
events_df["linked_company_ids_str"] = events_df["linked_company_ids"].apply(
    lambda x: "|".join(sorted(map(str, x)))
)

confidence_rank = {
    "high_sentence_level": 0,
    "medium_article_level": 1
}

events_df["confidence_rank"] = events_df["confidence"].map(confidence_rank).fillna(9)

# Deduplicate exact article-event-company candidates
events_dedup = (
    events_df
    .sort_values([
        "article_id",
        "event_type",
        "linked_company_ids_str",
        "confidence_rank"
    ])
    .drop_duplicates(
        subset=["article_id", "event_type", "linked_company_ids_str"],
        keep="first"
    )
    .reset_index(drop=True)
)

print("Events before deduplication:", len(events_df))
print("Events after deduplication:", len(events_dedup))

events_dedup.to_csv("company_event_candidates_dedup.csv", index=False)

Events before deduplication: 40577
Events after deduplication: 15166


In [19]:
events_pref_article = (
    events_dedup
    .sort_values([
        "article_id",
        "event_type",
        "confidence_rank"
    ])
    .drop_duplicates(
        subset=["article_id", "event_type"],
        keep="first"
    )
    .reset_index(drop=True)
)

events_pref_article.to_csv("company_event_pref_article_event.csv", index=False)

print("Preferred article-event rows:", len(events_pref_article))

print("\nPreferred event type counts:")
print(events_pref_article["event_type"].value_counts())

print("\nPreferred confidence counts:")
print(events_pref_article["confidence"].value_counts())

Preferred article-event rows: 13739

Preferred event type counts:
event_type
lawsuit_legal            3556
acquisition_merger       3494
earnings_results         2790
partnership_contract     1189
layoffs_restructuring     837
ipo_listing               823
dividend                  588
regulatory_approval       293
downgrade                 169
Name: count, dtype: int64

Preferred confidence counts:
confidence
medium_article_level    7103
high_sentence_level     6636
Name: count, dtype: int64


In [20]:
events_for_graph = events_dedup[
    events_dedup["confidence"] == "high_sentence_level"
].copy()

events_for_graph.to_csv("company_event_high_confidence_for_graph.csv", index=False)

print("Events for graph:", len(events_for_graph))

print("\nGraph event type counts:")
print(events_for_graph["event_type"].value_counts())

Events for graph: 7168

Graph event type counts:
event_type
acquisition_merger       2158
earnings_results         1894
lawsuit_legal            1280
partnership_contract      546
ipo_listing               442
dividend                  381
layoffs_restructuring     308
regulatory_approval        81
downgrade                  78
Name: count, dtype: int64


In [21]:
import ast
from collections import Counter
import pandas as pd

events_for_graph = pd.read_csv("company_event_high_confidence_for_graph.csv")

def parse_list_column(x):
    if isinstance(x, list):
        return x

    if pd.isna(x):
        return []

    try:
        value = ast.literal_eval(str(x))
        if isinstance(value, list):
            return value
        return []
    except Exception:
        return []

events_for_graph["linked_company_ids"] = events_for_graph["linked_company_ids"].apply(parse_list_column)
events_for_graph["linked_companies"] = events_for_graph["linked_companies"].apply(parse_list_column)

edge_counter = Counter()

for _, row in events_for_graph.iterrows():
    event_type = row["event_type"]

    for company_id in row["linked_company_ids"]:
        edge_counter[(company_id, event_type)] += 1

company_event_edges = pd.DataFrame([
    {
        "company_id": company_id,
        "event_type": event_type,
        "weight": weight
    }
    for (company_id, event_type), weight in edge_counter.items()
])

company_event_edges["company"] = company_event_edges["company_id"].map(company_id_to_name)

company_event_edges = company_event_edges.sort_values(
    "weight",
    ascending=False
)

company_event_edges.to_csv("company_event_edges_high_confidence.csv", index=False)

company_event_edges.head(20)

,company_id,event_type,weight,company
23,Q1472929,earnings_results,411,"Nasdaq, Inc."
24,Q1472929,acquisition_merger,101,"Nasdaq, Inc."
94,Q1472929,ipo_listing,92,"Nasdaq, Inc."
191,Q35476,acquisition_merger,73,AT&T
517,Q544847,acquisition_merger,71,Qualcomm
38,Q478214,lawsuit_legal,52,Tesla
46,Q301965,acquisition_merger,48,Sprint Corporation
62,Q35476,lawsuit_legal,47,AT&T
85,Q1472929,dividend,46,"Nasdaq, Inc."
17,Q780442,lawsuit_legal,45,Uber


In [22]:
event_type_summary = (
    events_for_graph["event_type"]
    .value_counts()
    .reset_index()
)

event_type_summary.columns = ["event_type", "event_mentions"]

event_type_summary

,event_type,event_mentions
0,acquisition_merger,2158
1,earnings_results,1894
2,lawsuit_legal,1280
3,partnership_contract,546
4,ipo_listing,442
5,dividend,381
6,layoffs_restructuring,308
7,regulatory_approval,81
8,downgrade,78


In [23]:
company_event_matrix = company_event_edges.pivot_table(
    index=["company_id", "company"],
    columns="event_type",
    values="weight",
    fill_value=0
).reset_index()

event_cols = [
    col for col in company_event_matrix.columns
    if col not in ["company_id", "company"]
]

company_event_matrix["total_event_mentions"] = company_event_matrix[event_cols].sum(axis=1)

company_event_matrix = company_event_matrix.sort_values(
    "total_event_mentions",
    ascending=False
)

company_event_matrix.to_csv("company_event_matrix_high_confidence.csv", index=False)

company_event_matrix.head(20)

event_type,company_id,company,acquisition_merger,dividend,downgrade,earnings_results,ipo_listing,lawsuit_legal,layoffs_restructuring,partnership_contract,regulatory_approval,total_event_mentions
185,Q1472929,"Nasdaq, Inc.",101.0,46.0,0.0,411.0,92.0,16.0,2.0,14.0,1.0,683.0
719,Q35476,AT&T,73.0,1.0,0.0,2.0,2.0,47.0,0.0,7.0,0.0,132.0
1013,Q544847,Qualcomm,71.0,0.0,0.0,4.0,0.0,33.0,0.0,7.0,7.0,122.0
1172,Q66,Boeing,12.0,1.0,0.0,7.0,0.0,31.0,5.0,33.0,0.0,89.0
731,Q3884,Amazon,34.0,0.0,0.0,9.0,3.0,18.0,6.0,16.0,0.0,86.0
681,Q312,Apple Inc.,19.0,2.0,2.0,15.0,2.0,30.0,0.0,14.0,0.0,84.0
1392,Q780442,Uber,21.0,0.0,0.0,1.0,10.0,45.0,0.0,5.0,0.0,82.0
822,Q478214,Tesla,1.0,0.0,14.0,3.0,2.0,52.0,0.0,8.0,0.0,80.0
1540,Q950380,CBS Corporation,32.0,1.0,0.0,1.0,1.0,29.0,0.0,4.0,0.0,68.0
1108,Q60238941,Fox Corporation,44.0,0.0,0.0,0.0,0.0,10.0,0.0,5.0,6.0,65.0


In [25]:
import pandas as pd

def sample_random_events(
    events_df,
    n=10,
    random_state=42,
    columns=None
):
    """
    Return a random sample of high-confidence event rows.

    Parameters
    ----------
    events_df : pandas.DataFrame
        Event dataframe containing a 'confidence' column.
    n : int
        Number of random events to sample.
    random_state : int
        Random seed for reproducibility.
    columns : list or None
        Optional list of columns to display.

    Returns
    -------
    pandas.DataFrame
        Random sample of high-confidence events.
    """

    high_confidence = events_df[
        events_df["confidence"] == "high_sentence_level"
    ].copy()

    sample_size = min(n, len(high_confidence))

    sampled = high_confidence.sample(
        n=sample_size,
        random_state=random_state
    )

    if columns is not None:
        available_columns = [col for col in columns if col in sampled.columns]
        sampled = sampled[available_columns]

    return sampled.reset_index(drop=True)

events_df = pd.read_csv("company_event_high_confidence_for_graph.csv")

events_for_review = sample_random_events(
    events_df,
    n=10,
    random_state=42,
    columns=[
        "article_id",
        "date",
        "title",
        "event_type",
        "trigger",
        "linked_companies",
        "sentence"
    ]
)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 50)
pd.set_option("display.width", None)
events_for_review

,article_id,date,title,event_type,trigger,linked_companies,sentence
0,48258,2018-05-08,Otelco Reports First Quarter 2018 Results,earnings_results,net income,['Otelco'],"OTELCO INC. CONDENSED CONSOLIDATED BALANCE SHEETS (in thousands, except share par value and share amounts) (unaudited) March 31, December 31, 2018 2017 Assets Current assets Cash and cash equivalents $ 6,288 $ 3,570 Accounts receivable: Due from subscribers, net of allowance for doubtful accounts of $235 and $206, respectively 4,489 4,647 Other 1,691 1,875 Materials and supplies 3,009 2,700 Prepaid expenses 1,451 3,122 Total current assets 16,928 15,914 Property and equipment, net 50,355 50,888 Goodwill 44,976 44,976 Intangible assets, net 1,223 1,328 Investments 1,506 1,632 Interest rate cap 34 - Other assets 190 201 Total assets $ 115,212 $ 114,939 Liabilities and Stockholders' Equity Current liabilities Accounts payable $ 1,315 $ 1,619 Accrued expenses 4,783 4,803 Advance billings and payments 1,632 1,684 Customer deposits 55 58 Current maturity of long-term notes payable, net of debt issuance cost 3,890 3,891 Total current liabilities 11,675 12,055 Deferred income taxes 18,939 18,939 Advance billings and payments 2,331 2,367 Other liabilities 17 13 Long-term notes payable, less current maturities and debt issuance cost 79,056 80,058 Total liabilities 112,018 113,432 Stockholders' equity Class A Common Stock, $.01 par value-authorized 10,000,000 shares; issued and outstanding 3,388,624 and 3,346,689 shares, respectively 34 34 Additional paid in capital 3,976 4,285 Accumulated deficit (816 ) (2,812 ) Total stockholders' equity 3,194 1,507 Total liabilities and stockholders' equity $ 115,212 $ 114,939\nOTELCO INC. AND SUBSIDIARIES CONDENSED CONSOLIDATED STATEMENTS OF OPERATIONS (in thousands, except share and per share amounts) (unaudited) Three Months Ended March 31, 2018 2017 Revenues $ 16,726 $ 17,380 Operating expenses Cost of services 7,965 7,813 Selling, general and administrative expenses 2,881 2,707 Depreciation and amortization 1,819 1,840 Total operating expenses 12,665 12,360 Income from operations 4,061 5,020 Other income (expense) Interest expense (1,459 ) (2,611 ) Other income 168 203 Total other expense (1,291 ) (2,408 ) Income before income tax expense 2,770 2,612 Income tax expense (774 ) (1,004 ) Net income $ 1,996 $ 1,608 Weighted average number of common shares outstanding: Basic 3,388,624 3,346,689 Diluted 3,420,181 3,444,370 Basic net income per common share $ 0.59 $ 0.48 Diluted net income per common share $ 0.58 $ 0.47\nOTELCO INC. AND SUBSIDIARIES CONDENSED CONSOLIDATED STATEMENTS OF CASH FLOWS (in thousands) (unaudited) Three Months Ended March 31, 2018 2017"
1,3427,2018-03-21,Altair Announces Fourth Quarter and Full Year 2017 Financial Results,earnings_results,financial results,"['Altair Engineering', 'Nasdaq, Inc.']",Altair Engineering Inc. (NASDAQ:ALTR) today announced its financial results for the fourth quarter and full 2017.
2,48201,2018-02-07,Tesco faces record 4 billion pound equal pay claim in Britain - BBC,lawsuit_legal,litigation,['Tesco Corporation'],"SLOW PROGRESS \nCrowley Woodford, employment partner at law firm Ashurst, said if the Tesco employees’ claim was successful “all major retailers, and indeed businesses more generally, could be exposed to a tidal wave of equal pay litigation.”"
3,12012,2018-05-02,Tableau Reports First Quarter 2018 Financial Results,earnings_results,net loss,"['Tableau Software, Inc.']","Tableau Software, Inc.\nCondensed Consolidated Statements of Operations\n(In thousands, except per share data)\n(Unaudited)\nThree Months Ended March 31,\n2018\n2017\nRevenues\nLicense\n$\n108,793\n$\n97,244\nMaintenance and services\n137,414\n102,662\nTotal revenues\n246,207\n199,906\nCost of revenues\nLicense\n3,954\n3,267\nMaintenance and services\n28,471\n23,388\nTotal cost of revenues (1)\n32,425\n26,655\nGross profit\n213,782\n173,251\nOperating expenses\nSales and marketing (1)\n138,406\n118,018\nResearc